# RNN으로 사인파의 다음 값 예측하기

이 노트북은 **시계열 생성 → 입력/정답 분리 → RNN 정의 → 학습 → 예측 곡선 비교 → 저장** 순서로 진행합니다.

`CELL` 값만 `rnn`, `gru`, `lstm`으로 바꾸면 같은 문제에서 세 순환 셀을 비교할 수 있습니다.

## 1. 라이브러리와 실행 설정

처음에는 빠른 설정으로 전체 흐름을 확인합니다. 정상 실행된 뒤 sample 수와 epoch를 늘리세요.

In [ ]:
from pathlib import Path
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
QUICK_RUN = True
CELL = 'rnn'  # 'rnn', 'gru', 'lstm'
EPOCHS = 10 if QUICK_RUN else 30
SAMPLES = 2_000 if QUICK_RUN else 5_000
SEQUENCE_LENGTH = 30
HIDDEN_SIZE = 64
BATCH_SIZE = 64
LEARNING_RATE = 2e-3
NOISE_STD = 0.03
OUTPUT_DIR = Path('/outputs/notebooks/rnn')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('PyTorch:', torch.__version__)
print('device:', DEVICE)
print('cell:', CELL)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. 사인파 시퀀스 만들기

각 sample은 사인파의 임의 위치에서 시작합니다. 31개 값을 만든 뒤 앞의 30개는 입력, 마지막 값은 정답으로 사용합니다. RNN 입력 shape는 `(batch, sequence, feature)`입니다.

In [ ]:
def make_dataset(samples, sequence_length, seed, noise_std):
    generator = torch.Generator().manual_seed(seed)
    starts = torch.rand(samples, generator=generator) * (12 * math.pi)
    offsets = torch.linspace(0, 2.5, sequence_length + 1)
    clean_waves = torch.sin(starts[:, None] + offsets[None, :])
    noise = torch.randn(clean_waves.shape, generator=generator) * noise_std
    waves = clean_waves + noise
    inputs = waves[:, :-1, None]
    targets = waves[:, -1]
    return inputs, targets

inputs, targets = make_dataset(SAMPLES, SEQUENCE_LENGTH, SEED, NOISE_STD)
split = int(len(inputs) * 0.8)
train_inputs, train_targets = inputs[:split], targets[:split]
test_inputs, test_targets = inputs[split:], targets[split:]

train_loader = DataLoader(
    TensorDataset(train_inputs, train_targets),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print('all inputs :', inputs.shape)
print('all targets:', targets.shape)
print('train/test :', len(train_inputs), '/', len(test_inputs))

## 3. 입력 30개와 정답 1개 보기

파란 점은 모델이 받는 입력이고 빨간 별은 맞혀야 할 다음 값입니다.

In [ ]:
sample_index = 0
x_values = np.arange(SEQUENCE_LENGTH)
plt.figure(figsize=(9, 3))
plt.plot(x_values, inputs[sample_index, :, 0], 'o-', label='input sequence')
plt.plot(SEQUENCE_LENGTH, targets[sample_index], 'r*', markersize=14, label='target')
plt.xlabel('time step')
plt.ylabel('value')
plt.title('One training sample')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. 순환 모델 정의

순환 계층은 매 시점의 hidden state를 `(B, 30, 64)`로 돌려줍니다. 모든 입력을 읽은 마지막 hidden state `(B, 64)`만 Linear 계층에 넣어 다음 값 하나를 예측합니다.

In [ ]:
class SequenceRegressor(nn.Module):
    def __init__(self, cell, hidden_size):
        super().__init__()
        cells = {'rnn': nn.RNN, 'gru': nn.GRU, 'lstm': nn.LSTM}
        self.recurrent = cells[cell](1, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        sequence, _ = self.recurrent(x)
        last_hidden = sequence[:, -1]
        return self.output(last_hidden).squeeze(-1)

model = SequenceRegressor(CELL, HIDDEN_SIZE).to(DEVICE)
print(model)
print('parameters:', sum(p.numel() for p in model.parameters()))

## 5. 순환 계층의 shape 직접 확인

작은 배치 네 개를 통과시키며 전체 sequence 출력과 최종 예측의 shape를 확인합니다.

In [ ]:
example_batch = train_inputs[:4].to(DEVICE)
sequence_output, recurrent_state = model.recurrent(example_batch)
predictions = model(example_batch)

print('input shape          :', tuple(example_batch.shape))
print('all hidden states    :', tuple(sequence_output.shape))
print('last hidden state    :', tuple(sequence_output[:, -1].shape))
print('final prediction     :', tuple(predictions.shape))
if CELL == 'lstm':
    print('LSTM h_n / c_n      :', tuple(recurrent_state[0].shape), tuple(recurrent_state[1].shape))
else:
    print('final recurrent state:', tuple(recurrent_state.shape))

## 6. MSE로 학습하기

숫자 클래스를 고르는 분류가 아니라 실수 하나를 예측하는 회귀이므로 평균 제곱 오차(MSE)를 사용합니다.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch_inputs, batch_targets in train_loader:
        batch_inputs = batch_inputs.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        batch_predictions = model(batch_inputs)
        loss = criterion(batch_predictions, batch_targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_inputs.size(0)

    epoch_loss = total_loss / len(train_inputs)
    train_losses.append(epoch_loss)
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f'epoch={epoch:03d} train_mse={epoch_loss:.6f}')

## 7. 학습 곡선 확인

MSE가 전반적으로 내려가는지 확인합니다. 매 epoch의 작은 흔들림은 mini-batch 순서가 무작위이기 때문에 자연스럽습니다.

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(range(1, EPOCHS + 1), train_losses, marker='o')
plt.xlabel('epoch')
plt.ylabel('train MSE')
plt.title(f'{CELL.upper()} training curve')
plt.grid(alpha=0.3)
plt.show()

## 8. 테스트 RMSE와 예측 비교

학습에 사용하지 않은 20% 데이터에서 예측합니다. RMSE는 target과 같은 단위라 MSE보다 직관적으로 읽을 수 있습니다.

In [ ]:
model.eval()
with torch.inference_mode():
    test_predictions = model(test_inputs.to(DEVICE)).cpu()

test_rmse = torch.sqrt(nn.functional.mse_loss(test_predictions, test_targets)).item()
print(f'test RMSE: {test_rmse:.6f}')

count = min(200, len(test_targets))
plt.figure(figsize=(12, 4))
plt.plot(test_targets[:count], label='target', linewidth=2)
plt.plot(test_predictions[:count], label='prediction', alpha=0.8)
plt.xlabel('test sample')
plt.ylabel('next value')
plt.title(f'{CELL.upper()} sine-wave next-value prediction')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 9. 모델 저장

모델 가중치, 설정, 손실 기록을 호스트의 `outputs/notebooks/rnn/`에 저장합니다.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / f'sine_{CELL}_notebook.pt'
torch.save({
    'model_state': model.state_dict(),
    'config': {
        'cell': CELL,
        'epochs': EPOCHS,
        'samples': SAMPLES,
        'sequence_length': SEQUENCE_LENGTH,
        'hidden_size': HIDDEN_SIZE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'noise_std': NOISE_STD,
        'seed': SEED,
    },
    'train_losses': train_losses,
    'test_rmse': test_rmse,
}, checkpoint_path)
print('saved:', checkpoint_path)

## 다음 실험

1. `CELL`만 `gru`, `lstm`으로 바꾸고 파라미터 수와 RMSE를 비교하세요.
2. `SEQUENCE_LENGTH`를 10, 30, 100으로 바꿔 기억 길이의 효과를 확인하세요.
3. `NOISE_STD`를 0.03에서 0.1로 높여 노이즈에 대한 변화를 확인하세요.
4. `HIDDEN_SIZE`를 16, 64, 128로 바꾸되 다른 값은 그대로 유지하세요.